# Lab 8 — Quantum Convolutional Neural Network (QCNN)

## Objectifs
- Implémenter une convolution quantique et un pooling quantique
- Construire une architecture QCNN hiérarchique
- Comparer avec un CNN classique

In [ ]:
import pennylane as qml
from pennylane import qnode
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

## Partie A — Convolution quantique

In [ ]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

def quantum_conv(weights, wires):
    """Couche de convolution quantique sur 2 qubits adjacents.
    Applique des rotations paramétrées + CNOT (partage de paramètres).
    """
    n_wires = len(wires)
    for i in range(n_wires):
        qml.RY(weights[0], wires=wires[i])
    for i in range(n_wires - 1):
        qml.CNOT(wires=[wires[i], wires[i + 1]])
    for i in range(n_wires):
        qml.RY(weights[1], wires=wires[i])
    for i in range(n_wires - 1):
        qml.CNOT(wires=[wires[i], wires[i + 1]])

print("Couche de convolution quantique définie (2 qubits, paramètres partagés)")
print("Structure : RY → CNOT → RY → CNOT")

dev_conv = qml.device("default.qubit", wires=2)

@qml.qnode(dev_conv)
def conv_demo(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(2))
    quantum_conv(weights, wires=range(2))
    return [qml.expval(qml.PauliZ(i)) for i in range(2)]

print(qml.draw(conv_demo)(np.array([0.5, 0.3]), np.array([[0.1, 0.2], [0.3, 0.4]])))

## Partie B — Pooling quantique

In [ ]:
def quantum_pool(source_wire, target_wire):
    """Couche de pooling quantique.
    Mesure le qubit source et effectue une rotation conditionnelle sur la cible.
    Réduit le nombre de qubits actifs de moitié.
    """
    qml.CNOT(wires=[source_wire, target_wire])
    qml.RY(np.pi / 4, wires=target_wire)

print("Couche de pooling quantique définie")
print("Opération : CNOT(source, target) → RY(π/4, target)")
print("Réduction : 4 qubits → 2 qubits → 1 qubit")

## Partie C — Architecture QCNN complète

In [ ]:
@qml.qnode(dev, interface="torch")
def qcnn_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))

    quantum_conv(weights[0], wires=[0, 1])
    quantum_conv(weights[1], wires=[2, 3])

    quantum_pool(0, 1)
    quantum_pool(2, 3)

    quantum_conv(weights[2], wires=[1, 3])

    quantum_pool(1, 3)

    return qml.expval(qml.PauliZ(3))

weight_shapes = {
    "weights": [(2, 2), (2, 2), (2, 2)]
}

qlayer = qml.qnn.TorchLayer(qcnn_circuit, weight_shapes=weight_shapes)

class QCNNModel(nn.Module):
    def __init__(self, qlayer):
        super().__init__()
        self.qlayer = qlayer

    def forward(self, x):
        return self.qlayer(x)

model_qcnn = QCNNModel(qlayer)

n_params = sum(p.numel() for p in model_qcnn.parameters() if p.requires_grad)
print(f"Architecture QCNN : {n_qubits} qubits")
print(f"Paramètres entraînables : {n_params}")
print(f"\nFlux : AngleEmbedding → Conv(0,1) Conv(2,3) → Pool → Conv(1,3) → Pool → Z(3)")

In [ ]:
X, y = make_classification(
    n_samples=400,
    n_features=n_qubits,
    n_informative=4,
    n_redundant=0,
    n_classes=2,
    random_state=42
)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X = scaler.fit_transform(X)

y = y.astype(np.float64) * 2 - 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

print(f"Dataset synthétique : {X.shape[0]} échantillons, {X.shape[1]} features")
print(f"Train : {len(X_train)}, Test : {len(X_test)}")
print(f"Distribution des classes : {np.unique(y, return_counts=True)}")

## Partie D — Entraînement et résultats

In [ ]:
n_epochs = 50
lr = 0.01

criterion = nn.MSELoss()
optimizer = optim.Adam(model_qcnn.parameters(), lr=lr)

train_losses = []
test_accuracies = []

for epoch in range(n_epochs):
    model_qcnn.train()
    optimizer.zero_grad()
    predictions = model_qcnn(X_train_t)
    loss = criterion(predictions, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model_qcnn.eval()
    with torch.no_grad():
        test_preds = model_qcnn(X_test_t)
        test_preds_binary = (test_preds > 0).float()
        y_test_binary = (y_test_t > 0).float()
        acc = accuracy_score(y_test_binary.numpy(), test_preds_binary.numpy())
        test_accuracies.append(acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} — Loss: {loss.item():.4f} — Acc: {acc:.4f}")

print(f"\nAccuracy finale QCNN : {test_accuracies[-1]:.4f}")

## Partie E — Comparaison avec CNN classique

In [ ]:
class ClassicalCNN(nn.Module):
    """CNN classique équivalent en profondeur au QCNN.
    Conv1D(4,4) → Pool → Conv1D(4,2) → Pool → Linear(2,1)
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 4, kernel_size=2, padding=1)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(4, 2, kernel_size=2, padding=1)
        self.pool2 = nn.MaxPool1d(2)
        self.fc = nn.Linear(2, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x.squeeze()

model_cnn = ClassicalCNN()

n_params_cnn = sum(p.numel() for p in model_cnn.parameters())
n_params_qcnn = sum(p.numel() for p in model_qcnn.parameters())

print(f"Paramètres QCNN : {n_params_qcnn}")
print(f"Paramètres CNN classique : {n_params_cnn}")

In [ ]:
criterion_cnn = nn.MSELoss()
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=lr)

cnn_losses = []
cnn_accuracies = []

for epoch in range(n_epochs):
    model_cnn.train()
    optimizer_cnn.zero_grad()
    preds = model_cnn(X_train_t)
    loss = criterion_cnn(preds, y_train_t)
    loss.backward()
    optimizer_cnn.step()
    cnn_losses.append(loss.item())

    model_cnn.eval()
    with torch.no_grad():
        test_preds = model_cnn(X_test_t)
        test_preds_binary = (test_preds > 0).float()
        y_test_binary = (y_test_t > 0).float()
        acc = accuracy_score(y_test_binary.numpy(), test_preds_binary.numpy())
        cnn_accuracies.append(acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} — Loss: {loss.item():.4f} — Acc: {acc:.4f}")

print(f"\nAccuracy finale CNN classique : {cnn_accuracies[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label="QCNN", color="purple")
axes[0].plot(cnn_losses, label="CNN classique", color="blue")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Courbe de perte")
axes[0].legend()

axes[1].plot(test_accuracies, label="QCNN", color="purple")
axes[1].plot(cnn_accuracies, label="CNN classique", color="blue")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Courbe d'accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

print("=" * 45)
print(f"{'Métrique':<20} {'QCNN':<12} {'CNN':<12}")
print("=" * 45)
print(f"{'Accuracy':<20} {test_accuracies[-1]:<12.4f} {cnn_accuracies[-1]:<12.4f}")
print(f"{'Paramètres':<20} {n_params_qcnn:<12} {n_params_cnn:<12}")
print("=" * 45)

## Exercices
1. Augmenter la profondeur du QCNN (3 couches de conv+pooling)
2. Tester avec un encodeur IQP au lieu d'AngleEmbedding
3. Comparer le nombre de paramètres QCNN vs CNN classique